In [5]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/dd/tpdzgsps29l6xqx8hm7z2_rr0000gp/T/tmpif6g6osu


In [6]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
# Load env vars

from dotenv import load_dotenv

load_dotenv()

True

In [8]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [9]:
from anthropic.types import MessageParam
from collections.abc import Iterable

def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text }
    messages.append(assistant_message)

from anthropic.types import TextBlock, Message

def get_message_text(message: Message) -> str:
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "" # empty string if none
    )

def chat(messages: Iterable[MessageParam], system: str|None = None, temperature = 1.0, stop_sequences: list[str] = []) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_message_text(message)

In [ ]:
import json
from typing import TypedDict

class TaskObj(TypedDict):
    task: str

class GradedTestCaseObj(TypedDict):
    test_case: TaskObj
    output: str
    score: int
    reasoning: str

with open('data/007_generate_eval_dataset-tasks.json', 'r', encoding='utf-8') as file:
    dataset: list[TaskObj] = json.load(file)

print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI (e.g., 's3://my-bucket-us-east-1/path'). The function should return the region code or None if not found."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'. Include the necessary Version, Statement, Effect, Action, and Resource fields."}, {'task': "Write a regular expression that matches valid AWS EC2 instance IDs (format: i- followed by 17 hexadecimal characters, e.g., 'i-0abcd1234efgh5678')."}]


In [11]:
def run_prompt(test_case: TaskObj) -> str:
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [ ]:
class EvalReturnObj(TypedDict):
    score: int
    reasoning: str

def grade_by_model(test_case: TaskObj, output: str) -> EvalReturnObj:
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task:
    <task>
    {test_case["task"]}
    </task>

    Solution:
    <solution>
    {output}
    </solution>
    
    Provide your evaluation as a structured JSON object like this:
    ```json
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    ```
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    res: EvalReturnObj = json.loads(eval_text)
    return res


In [ ]:
def run_test_case(test_case: TaskObj) -> GradedTestCaseObj:
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    grade = grade_by_model(test_case, output)

    result: GradedTestCaseObj = {
        "output": output,
        "test_case": test_case,
        "score": grade["score"],
        "reasoning": grade["reasoning"]
    }
    
    return result

def run_eval(dataset: Iterable[TaskObj]) -> list[GradedTestCaseObj]:
    """Loads the dataset and calls run_test_case with each case"""
    results: list[GradedTestCaseObj] = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [14]:
results = run_eval(dataset)

In [15]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_aws_region_from_s3_uri(s3_uri: str) -> Optional[str]:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URI.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/path' or 's3://bucket-name'\n        \n    Returns:\n        AWS region code (e.g., 'us-east-1') or None if not found\n        \n    Examples:\n        >>> extract_aws_region_from_s3_uri('s3://my-bucket-us-east-1/path')\n        'us-east-1'\n        >>> extract_aws_region_from_s3_uri('s3://bucket-eu-west-2')\n        'eu-west-2'\n        >>> extract_aws_region_from_s3_uri('s3://my-bucket/path')\n        None\n    \"\"\"\n    \n    # Valid AWS region pattern\n    # Regions follow format: area-direction-number (e.g., us-east-1, eu-west-2, ap-southeast-1)\n    aws_region_pattern = r'(us|eu|ap|ca|sa|me|af|cn)-(north|south|east|west|central)?-?\\d+'\n    \n    # Extract bucket name fr